In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import torch
import torch.nn as nn
import seaborn as sns
import os
from dataProcesser import read_file_high_level, read_file_low_level , dicts_to_dataframe
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import clear_output
import time
import joblib
# plt.style.use('ggplot')  # Using a standard Matplotlib style
# sns.set_palette("Set2")

processes = ['assembly', 'electronics', 'material-join', 'material-process']
device = 'cpu'

import numpy as np
import torch
import random

def set_seeds(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
set_seeds()
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
variable = "sample_size"
# print(f"Current CUDA device: {torch.cuda.current_device()}")
# print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

In [ ]:
class BaselineModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BaselineModel, self).__init__()
        self.ff_network = nn.Sequential(
            nn.Linear(input_dim, int(input_dim*1.5)),
            nn.ReLU(),
            nn.Linear(int(input_dim*1.5), input_dim*2),
            nn.ReLU(),
            nn.Linear(input_dim*2, input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, int(input_dim/2) if input_dim/2>output_dim else output_dim),
            nn.ReLU(),
            nn.Linear(int(input_dim/2) if input_dim/2>output_dim else output_dim, output_dim)
        )

    def forward(self, x):
        return self.ff_network(x)

In [ ]:
def train_model(train_df, covariates, treatment, outcome, epochs, batch_size, val_df=None, plot=True, lr=0.001):
    
    # use cuda if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training the model on {device}")

    input_dim = len(covariates)
    output_dim = len(outcome)

    # print(input_dim)
    model = BaselineModel(input_dim + 1, output_dim)
    # print(model)
    
    model.to(device)

    # shuffle the training data
    train_df = train_df.sample(frac=1).reset_index(drop=True)
    X = train_df[covariates].values
    T = train_df[treatment].values
    Y = train_df[outcome].values
    X_T = np.concatenate([X, T.reshape(-1, 1)], axis=1)
    print(X_T.shape)
    
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    #         optimizer,
    #         T_0=10,  # First restart occurs after 10 epochs
    #         T_mult=2,  # Each restart cycle is twice as long
    #         eta_min=1e-6  # Minimum learning rate
    #     )
    
    
    X_T = torch.tensor(X_T, dtype=torch.float32).to(device)
    Y = torch.tensor(Y, dtype=torch.float32).reshape(-1, output_dim).to(device)

    DataLoader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_T, Y), batch_size=batch_size, shuffle=True)
    
    
    # Prepare validation data if provided
    if val_df is not None:
        X_val = val_df[covariates].values
        T_val = val_df[treatment].values
        Y_val = val_df[outcome].values
        X_T_val = np.concatenate([X_val, T_val.reshape(-1, 1)], axis=1)
        X_T_val = torch.tensor(X_T_val, dtype=torch.float32)
        Y_val = torch.tensor(Y_val, dtype=torch.float32).reshape(-1, 1)
    
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for i, (X_T_batch, Y_batch) in enumerate(DataLoader):
            optimizer.zero_grad()
            output = model(X_T_batch)
            loss = loss_fn(output, Y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_train_loss = epoch_loss / len(DataLoader)
        train_losses.append(avg_train_loss)
        # scheduler.step()
        
        # Validation step
        if val_df is not None:
            model.eval()
            with torch.no_grad():
                val_output = model(X_T_val)
                val_loss = loss_fn(val_output, Y_val)
                val_losses.append(val_loss.item())

        if ((epoch + 1) % 100 == 0) and plot == True:
            clear_output(wait=False)
            plt.clf()  # Clear the figure
            plt.figure(figsize=(10, 5))
            plt.plot(range(1, epoch + 2), train_losses, label='Training Loss')
            if val_df is not None:
                plt.plot(range(1, epoch + 2), val_losses, label='Validation Loss')
            plt.xlabel('Epochs')
            plt.ylabel('Loss')
            plt.title('Training and Validation Loss')
            plt.legend()
            plt.show()    
    # return model, train_losses, val_losses
    return model

In [ ]:
def assign_tree_depth(df1, df2):
    # Assuming df1 has 'query_id' and 'tree_depth' columns
    # and df2 has a 'query_id' column
    
    # Create a dictionary mapping query_id to tree_depth
    depth_dict = dict(zip(df1['query_id'], df1['tree_depth']))
    
    # Assign tree_depth to df2 based on query_id
    df2['tree_depth'] = df2['query_id'].map(depth_dict)
    
    return df2

def common_member(a, b):
    a_set = set(a)
    b_set = set(b)
 
    if (a_set & b_set):
        return list(a_set & b_set)
    else:
        return None
    
def get_pred(x, covariates, treatment, model):
    X = x[covariates].values
    T = x[treatment].values
    X_T = np.concatenate([X, T.reshape(-1, 1)], axis=1)
    # print(X_T.shape)
    X_T = torch.tensor(X_T, dtype=torch.float32).to(device)
    with torch.no_grad():
        predicted_effect = model(X_T).cpu().numpy()

    pred = []
    for effect in predicted_effect:
        pred.append(effect)
    return pred

def get_gt (x, outcome):
    return x[outcome]

In [ ]:
df_lowLevel = []
df_HighLevel = pd.read_csv('csvs/factory_high_level.csv')

senarios = list(range(1, 51))
demands = list(range(5, 1005, 5))
workers = ['00', '01']

for scenario in tqdm(senarios):
    for demand in demands:
        for worker in workers:
            simulation_result = f'simulation_report/worker_{worker}/scenario_{scenario}/scenario_{scenario}_demand_{demand}_report.json'
            df_lowLevel.extend(read_file_low_level (simulation_result))

df_lowLevel = dicts_to_dataframe(df_lowLevel)
df_lowLevel['tree_depth'] =  0

df_lowLevel = assign_tree_depth(df_HighLevel, df_lowLevel)
df_lowLevel.head()

In [ ]:
# 'process_runtime', 
outcome = ['parts_produced_by_process']
covariates = ['material-join_part', 'electronics_part', 'electronic_component', 'misc_component', 'fastener', 'material-process_part', 'raw_material', 'assembly_part']
treatment = 'treatment_id'

os.makedirs('Models/Pytorch/LowLevel', exist_ok=True)

with open('Models/Pytorch/LowLevel/outcome.json', "w") as f:
    json.dump(outcome, f)

os.makedirs('Models/Scalers/LowLevel', exist_ok=True)

In [ ]:
processes = list(set(df_lowLevel.process))
for process in tqdm(processes):
    df_low = df_lowLevel[df_lowLevel.process == process].copy(deep=True)
    df_low = df_low.loc[:, (df_low != 0).any(axis=0)]
    covs = common_member(covariates, df_low.columns)
    os.makedirs(f'Models/Pytorch/LowLevel/{process}', exist_ok=True)
    with open(f'Models/Pytorch/LowLevel/{process}/covariates.json', "w") as f:
        json.dump(covs, f)
    df_low.to_csv(f'csvs/{process}_low_level.csv', index=False)

In [ ]:
# df_low = pd.read_csv('csvs/assembly_low_level.csv')
process = processes[1]

print(process)
df_low = pd.read_csv(f'csvs/{process}_low_level.csv')
rows2drop =['assembly_part', 'electronics_part', 'treatment_id', 'fastener', 'material-process_part', 'material-join_part', 'misc_component', 'tree_depth', 'raw_material', 'electronic_component', ]

df_low = df_low.loc[:, (df_low != 0).any(axis=0)]
# df_low['throughput'] = df_low['parts_produced_by_process']/df_low['process_runtime']
rows2drop = list(set(rows2drop) & set(df_low.columns))
df_numeric = df_low.select_dtypes(include=['float64', 'int64'])

corr = df_numeric.corr().drop(rows2drop)
corr = corr.drop(['process_runtime', 'parts_produced_by_process'], axis=1)

plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(20, 5))
sns.heatmap(corr)
print(outcome)
df_low.head()

# display(corr)

In [ ]:
outcome = json.load(open(f'Models/Pytorch/LowLevel/outcome.json'))
variable = "tree_depth"
for process in tqdm(processes):
    df_low = pd.read_csv(f'csvs/{process}_low_level.csv')
    covs = json.load(open(f'Models/Pytorch/LowLevel/{process}/covariates.json'))
    columns2scale = covs.copy()
    # columns2scale.extend(outcome_lowLevel)
    scalerIn = MinMaxScaler()
    scalerOut = StandardScaler()
    df_low[columns2scale] = scalerIn.fit_transform(df_low[columns2scale])
    df_low[outcome] = scalerOut.fit_transform(df_low[outcome])

    os.makedirs(f'Models/Scalers/LowLevel/{process}', exist_ok=True)
    joblib.dump(scalerIn, f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Input_{variable}.pkl')
    joblib.dump(scalerOut, f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Outcome_{variable}.pkl')
    # # df_low.describe()
    # train_low, test_low = train_test_split (df_low, test_size=0.2, random_state=420)
    # model = train_model(train_low, covs, treatment, outcome_lowLevel, epochs=100, batch_size=512, plot=True)

    print(process)

In [ ]:
def get_train_test(df_low, knob_value, variable):
    print(knob_value, variable)
    hl_model_path = 'Models/Pytorch/HighLevel'
    with open(f'{hl_model_path}/{variable}/train_qids_{variable}_{knob_value}.json', 'r') as f:
        train_qids = json.load(f)
    with open(f'{hl_model_path}/{variable}/test_qids_{variable}_{knob_value}.json', 'r') as f:
        test_qids = json.load(f)
    with open(f'{hl_model_path}/{variable}/treatment_assignments_{variable}_{knob_value}.json', 'r') as f:
        treatment_assignments = json.load(f)
    print(len(train_qids), len(test_qids), len(treatment_assignments))
    print(test_qids)
    test = df_low[df_low['query_id'].isin(test_qids)]
    train = df_low[df_low['query_id'].isin(train_qids)]
    print(len(train), len(test))
    train["treatment_assigned"] = train["query_id"].map(treatment_assignments)
    train = train[train["treatment_assigned"] == train["treatment_id"]]
    train = train.drop(columns=["treatment_assigned"])
    print(len(train), len(test))
    
    return train, test

In [ ]:
def get_scaler_feature_names(scaler):
    """
    Extract column names from a fitted scikit-learn scaler object.
    
    Parameters:
    scaler: A fitted scikit-learn scaler (StandardScaler, MinMaxScaler, RobustScaler, etc.)
    
    Returns:
    list: List of feature names used during fitting
    
    Raises:
    ValueError: If scaler is not fitted or feature names are not available
    """
    if not hasattr(scaler, 'n_features_in_'):
        raise ValueError("Scaler is not fitted yet. Please fit the scaler first.")
    
    # Try different attribute names where feature names might be stored
    feature_names = None
    
    # Check if feature names were passed during fit
    if hasattr(scaler, 'feature_names_in_'):
        feature_names = list(scaler.feature_names_in_)
    
    # For older scikit-learn versions or if names weren't passed
    elif hasattr(scaler, '_feature_names_in'):
        feature_names = list(scaler._feature_names_in)
    
    # If no feature names found, return generic names
    if feature_names is None:
        feature_names = [f'feature_{i}' for i in range(scaler.n_features_in_)]
    
    return feature_names

In [ ]:
# Sample size experiment

In [ ]:
import json
from sklearn.model_selection import train_test_split
lr = 0.001
train_test_split_type = "tree_depth"
obs_biasing_type = "random"

knob_values = [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
test_qids = json.load(open(f'Models/Pytorch/HighLevel/{variable}/test_qids_{variable}.json'))
outcome = json.load(open(f'Models/Pytorch/LowLevel/outcome.json'))
for process in tqdm(processes):
    df_low = pd.read_csv(f'csvs/{process}_low_level.csv')
    covs = json.load(open(f'Models/Pytorch/LowLevel/{process}/covariates.json'))
    scalerIn = joblib.load(f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Input_{variable}.pkl')
    scalerOut = joblib.load(f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Outcome_{variable}.pkl')
    df_low[covs] = scalerIn.transform(df_low[covs])
    df_low[outcome] = scalerOut.transform(df_low[outcome])
    
    for knob_value in knob_values:
        train, test = get_train_test(df_low, 0, 'bias_strength')
        print(len(train), len(test))
        train_qids = json.load(open(f'Models/Pytorch/HighLevel/{variable}/train_qids_{variable}_{knob_value}.json'))
        train = train[train['query_id'].isin(train_qids)]
        print(len(train), len(test))
        train, val = train_test_split(train, test_size=0.2, random_state=420)
        flag = 'n'
        while flag.lower() != 'y':
            model = train_model(train, covs, treatment, outcome, epochs=100, batch_size=1000, plot=True, lr=lr, val_df=val)
            
            print (f'Model trained for {process} with {variable}={knob_value}')
            flag = input('Are you happy with the plot? (y/n) ')
            if flag.lower() == 'y':
                torch.save(model, f'Models/Pytorch/LowLevel/{process}/LowLevelModel_{variable}_{knob_value}.pt')
                break
            else:
                # maybe modify the learning rate
                lr = float(input('Enter the new learning rate: '))
                continue

# Compositional generalization and observational bias experiment 

In [ ]:
import json
from sklearn.model_selection import train_test_split
lr = 0.001
# For CG experiment, use the following values
train_test_split_type = "tree_depth"
obs_biasing_type = "random"

# For Observational bias experiment, use the following values
# train_test_split_type = "bias_strength"
# obs_biasing_type = "tree_depth"
if train_test_split_type == "tree_depth":
    knob_values = [3, 4, 5, 6, 7, 8]
    variable = 'tree_depth'
else:
    knob_values = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    variable = 'bias_strength'
outcome = json.load(open(f'Models/Pytorch/LowLevel/outcome.json'))
for process in tqdm(processes):
    df_low = pd.read_csv(f'csvs/{process}_low_level.csv')
    # covs = json.load(open(f'Models/Pytorch/LowLevel/{process}/covariates.json'))
    scalerIn = joblib.load(f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Input_{variable}.pkl')
    scalerOut = joblib.load(f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Outcome_{variable}.pkl')
    covs = list(get_scaler_feature_names(scalerIn))
    df_low[covs] = scalerIn.transform(df_low[covs])
    df_low[outcome] = scalerOut.transform(df_low[outcome])
    train, test = train_test_split (df_low, test_size=0.2, random_state=420)
    for knob_value in knob_values:
        train, test = get_train_test(df_low, knob_value, variable)
        print(train["tree_depth"].value_counts())
        train, val = train_test_split(train, test_size=0.2, random_state=420)
        flag = 'n'
        # Currently, we are manually tuning the learning rate for different processes and train/test splits 
        # TODO: Automate this process using a grid search or a similar technique
        while flag.lower() != 'y':
            model = train_model(train, covs, treatment, outcome, epochs=100, batch_size=1000, plot=True, lr=lr, val_df=val)
            
            print (f'Model trained for {process} with {variable}={knob_value}')
            flag = input('Are you happy with the plot? (y/n) ')
            if flag.lower() == 'y':
                torch.save(model, f'Models/Pytorch/LowLevel/{process}/LowLevelModel_{variable}_{knob_value}.pt')
                break
            else:
                # maybe modify the learning rate
                lr = float(input('Enter the new learning rate: '))
                continue

# Plot performance of the trained low-level models

In [ ]:
train_test_split_type = "tree_depth"
obs_biasing_type = "random"
markersize = 30
linewidth = 7
font_size = 65
plt.rcParams.update({'font.size': font_size})
plt.rcParams.update({'legend.fontsize': font_size})
plt.rcParams.update({'axes.labelsize': font_size})
plt.rcParams.update({'axes.titlesize': font_size})
sns.set_style("whitegrid")
variable = 'tree_depth'
knob_values = [3, 4, 5, 6, 7, 8]
num_outcomes = len(outcome)
# fig, axs = plt.subplots(1, 1 * num_outcomes, (25,25))
fig = plt.figure(figsize=(25, 25))
# fig.suptitle('R2 and MSE for Train and Test Sets Across Processes and Tree Depths')

for i, metric in enumerate(['R2']):
    for j, dataset in enumerate(['Test']):
        for k, out in enumerate(outcome):
            for process in processes:
                df_low = pd.read_csv(f'csvs/{process}_low_level.csv')
                # covs = json.load(open(f'Models/Pytorch/LowLevel/{process}/covariates.json'))
                scalerIn = joblib.load(f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Input.pkl')
                scalerOut = joblib.load(f'Models/Scalers/LowLevel/{process}/LowLevelScaler-Outcome.pkl')
                covs = list(get_scaler_feature_names(scalerIn))
                print(covs)
                print(processes)
                df_low[covs] = scalerIn.transform(df_low[covs])
                df_low[outcome] = scalerOut.transform(df_low[outcome])
                # train, test = train_test_split(df_low, test_size=0.2, random_state=420)
                train, test = get_train_test(df_low, knob_value, variable)
                depths = list(set(df_low.tree_depth))
                metric_values = []
                
                for knob_value in knob_values:
                    # model = torch.load(f'Models/Pytorch/LowLevel/{process}/LowLevelModel_{variable}_{knob_value}.pt', weights_only=False, map_location=torch.device('cpu'))
                    model = torch.load(f'Models/Pytorch/LowLevel/{process}/{process}_depth_{knob_value}.pth', weights_only=False, map_location=torch.device('cpu'))
                    pred_train = pd.DataFrame(get_pred(train, covs, treatment, model), columns=outcome)
                    gt_train = get_gt(train, outcome)
                    pred_test = pd.DataFrame(get_pred(test, covs, treatment, model), columns=outcome)
                    gt_test = get_gt(test, outcome)
                    
                    if metric == 'R2':
                        train_metric = r2_score(gt_train[out], pred_train[out])
                        test_metric = r2_score(gt_test[out], pred_test[out])
                    else:  # MSE
                        train_metric = mean_squared_error(gt_train[out], pred_train[out])
                        test_metric = mean_squared_error(gt_test[out], pred_test[out])
                    
                    metric_values.append(train_metric if dataset == 'Train' else test_metric)
                
                col_index = j * num_outcomes + k
                print(i,j)
                # axs = fig.add_subplot(1, 1 * num_outcomes, col_index + 1)
                plt.plot(knob_values, metric_values, label=process, marker='o', markersize=markersize, linewidth=linewidth)
                plt.xlabel('Tree Depth')
                plt.ylabel(r'$ R^2$' if metric == 'R2' else 'MSE')
                plt.ylim([0, 1.1] if metric == 'R2' else [-0.1, 0.6])
                plt.title(f'Performance of Component Models Across {variable} Values')
                plt.legend()
                # axs.plot(train_depths, metric_values, label=process, marker='o')
                # axs.set_xlabel('Tree Depth')
                # axs.set_ylabel(metric)
                # axs.set_ylim([0, 1.1] if metric == 'R2' else [-0.1, 0.6])
                # axs.set_title(f'{metric} for {dataset} Set - {out}')
                # axs.legend()
                
            #     axs[i, col_index].plot(train_depths, metric_values, label=process, marker='o')
            
            # col_index = j * num_outcomes + k
            # axs[i, col_index].set_xlabel('Tree Depth')
            # axs[i, col_index].set_ylabel(metric)
            # axs[i, col_index].set_ylim([0, 1.1] if metric == 'R2' else [-0.1, 0.6])
            # axs[i, col_index].set_title(f'{metric} for {dataset} Set - {out}')
            # axs[i, col_index].legend()
            # axs[i, col_index].grid(True)

plt.tight_layout()
plt.show()